# GardnAIze — 06 Evaluate and export ResNet-18

Loads locally trained weights, evaluates on a held-out dataset and prints a classification report/confusion matrix. Do not claim accuracy if test data is missing or contaminated.

Expected independent test directory: `datasets/images/garden_test/{garden,not_garden,unusable}/`. **Never use training images in this directory.**

In [ ]:
import json
from pathlib import Path
import torch
from torch import nn
from torchvision import models, datasets
from torch.utils.data import DataLoader
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
ROOT=Path.cwd()
if ROOT.name=="notebooks":ROOT=ROOT.parent
OUT=ROOT/"models"/"weights"
TEST=ROOT/"datasets"/"images"/"garden_test"
metadata=json.loads((OUT/"garden_resnet18_v1.json").read_text())
classes=metadata["classes"]
assert TEST.exists(),"Create an independent labeled garden_test dataset first."
weights=models.ResNet18_Weights.DEFAULT
model=models.resnet18(weights=None)
model.fc=nn.Linear(model.fc.in_features,len(classes))
model.load_state_dict(torch.load(OUT/"garden_resnet18_v1.pth",map_location="cpu",weights_only=True))
model.eval()
dataset=datasets.ImageFolder(TEST,transform=weights.transforms())
assert dataset.classes==classes, f"Expected {classes}; found {dataset.classes}"
loader=DataLoader(dataset,batch_size=8,shuffle=False,num_workers=0)
true=[];pred=[]
with torch.inference_mode():
    for images,labels in loader:
        logits=model(images)
        true.extend(labels.tolist());pred.extend(logits.argmax(1).tolist())
print(classification_report(true,pred,labels=list(range(len(classes))),target_names=classes,zero_division=0))
ConfusionMatrixDisplay(confusion_matrix(true,pred,labels=list(range(len(classes)))),display_labels=classes).plot(cmap="Blues")
plt.show()

## Deployment contract

Proposed FastAPI endpoint: `POST /predict/garden-photo` with multipart image input. Return `model_version`, `predicted_class`, `confidence`, and `class_probabilities`; reject invalid MIME/oversized files. Load weights once at startup from a read-only Docker volume. Add an abstention threshold calibrated on validation data. Do not infer soil type from the image.

## Jury evidence

Record data provenance, dataset split, ResNet-18 residual blocks, frozen backbone vs trained fully connected head, cross-entropy loss, optimizer, confusion matrix, latency, model hash and limitations. This notebook does **not** deploy an API; integration is a separate step.